<center><h1>Select and Load Data</h1></center>

## Import packages
Dependency Specification: pandas==3.0.5, duckdb==1.5.6

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

from utils import *

## Select Features
Knowing what features to include is based on domain knowledge of League of Legends Esports. Refer to the original source database by Oracke's Elixir by Tim Sevenhuysen.

In [21]:
cols_selected = [
                # Identifiers
                'league', 'split', 'date', 'side', 'position', 'playername', 'teamname', 
                # Draft Info
                'firstPick', 'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 
                # Player Stats
                'doublekills', 'triplekills', 'quadrakills', 'pentakills', 
                'firstbloodkill', 'firstbloodassist', 'firstbloodvictim',
                'wardskilled', 'visionscore',
                # Team Stats
                'result', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 
                'elders', 'opp_elders', 'firstherald', 'void_grubs', 'opp_void_grubs',
                'firstbaron', 'barons', 'opp_barons', 'firsttower', 'towers', 'opp_towers',
                # Team and Player Stats
                'kills', 'assists', 'dpm', 'totalgold', 
                'killsat10', 'assistsat10', 'opp_killsat10', 'opp_deathsat10',
                'killsat15', 'assistsat15', 'opp_killsat15', 'opp_deathsat15',
                'killsat20', 'assistsat20', 'opp_killsat20', 'opp_deathsat20',
                'killsat25', 'assistsat25', 'opp_killsat25', 'opp_deathsat25',
                # Ingame Stats
                'golddiffat10', 'xpdiffat10', 'golddiffat15', 'xpdiffat15', 
                'golddiffat20', 'xpdiffat20', 'golddiffat25', 'xpdiffat25'
                 ]
cols_selected = [f'"{col}"' for col in cols_selected] # quote the cols for a smooth join

year_selected = ["2026"] # temporarily hardcoded df year

## Load Data
Load data from pre-downloaded Oracle Elixer's Data from 2014 to 2026 to a local persistent database file for quick access on future sessions. This is for the main model

*Run the code below to create or update the duckdb file.*

In [22]:
# Connect to persistent database file on disk
con = duckdb.connect("lol_predictors.duckdb")

oe_data_path = Path("./OE datasets")

for dataset in oe_data_path.iterdir():
    if dataset.is_file() and dataset.suffix == ".csv" and dataset.stem[:4] in year_selected:
        var_name = f"df_{dataset.stem[:4]}" # e.g., df_2014
        
        # Read CSV via pandas
        df = pd.read_csv(dataset, low_memory=False)
        
        # Save directly into a permanent DuckDB table in your file!
        con.execute(
            f"""CREATE OR REPLACE TABLE {var_name} AS SELECT {", ".join(cols_selected)} 
            FROM df
            WHERE datacompleteness  = 'complete'""")
        print(f"Saved {var_name} into DuckDB file.")

con.close()

Saved df_2026 into DuckDB file.
